In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import polars as pl
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

/home/aditya/dev/prod-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Download dataset from kaggle
# !curl -L -o /content/data/genius-lyrics.zip\
#   https://www.kaggle.com/api/v1/datasets/download/carlosgdcj/genius-song-lyrics-with-language-information
# !unzip /content/data/genius-lyrics.zip -d /content/data/
# !rm /content/data/genius-lyrics.zip

In [3]:
# CSV_PATH = "../data/lyrics/song_lyrics.csv"
CSV_PATH = "../data/lyrics/song_lyrics_the_strokes.csv"
BATCH_SIZE = 10000
CHROMA_PATH = "../db/chroma/genius-lyrics-the-strokes"
# MILVUS_URI = "http://localhost:19530"

In [4]:

size_gb = os.path.getsize(CSV_PATH) / (1024 ** 3)
print(f"{size_gb:.4f} GB")

0.0001 GB


In [5]:
lf = pl.scan_csv(CSV_PATH)
lf.collect_schema()

Schema([('title', String),
        ('tag', String),
        ('artist', String),
        ('year', Int64),
        ('views', Int64),
        ('features', String),
        ('lyrics', String),
        ('id', Int64),
        ('language_cld3', String),
        ('language_ft', String),
        ('language', String)])

In [6]:
# Number of rows
ROWS = lf.select(pl.len()).collect().item()
ROWS

101

In [7]:
import re

def split_lyrics(lyrics):
    chunks = [
        c.strip()
        for c in re.split(r"\n\n", lyrics)
        if c.strip()
    ]
    return chunks

def chunk_lyrics_by_section(lyrics):
    # Split by section headers like [Verse 1], [Chorus], etc.
    sections = re.split(r'(\[[^\]]+\])', lyrics)
    
    chunks = []
    for i in range(1, len(sections), 2):  # headers are at odd indices
        if i + 1 < len(sections):
            header = sections[i]
            content = sections[i + 1].strip()
            if content:  # Skip empty sections
                chunk = f"{header}\n{content}"
                chunks.append(chunk)
    
    return chunks

all_lyrics = lf.select('lyrics').collect()
example_lyric = (all_lyrics[1]['lyrics'][0])
print(split_lyrics(example_lyric))
print(chunk_lyrics_by_section(example_lyric)) # better

["[Verse 1]\nIn many ways, they'll miss the good old days\nSomeday, someday\nYeah, it hurts to say, but I want you to stay\nSometimes, sometimes\nWhen we was young, oh man, did we have fun\nAlways, always\nPromises, they break before they're made\nSometimes, sometimes", "[Chorus]\nOh, Maya says I'm lacking in depth\nI will do my best\nYou say you wanna stand by my side\nDarling, your head's not right\nI see alone we stand; together we fall apart\nYeah, I think I'll be alright\nI'm working so I won't have to try so hard\nTables, they turn sometimes\nOh, someday\nNo, I ain't wasting no more time\n[Interlude]\n(Trying, trying)", '[Verse 2]\nAnd now my fears, they come to me in threes\nSo I sometimes\nSay, "Fate, my friend, you say the strangest things\nI find sometimes"', "[Chorus]\nOh, Maya says I'm lacking in depth\nShit, I will try my best\nYou say you wanna stay by my side\nDarling, your head's not right\nI see alone we stand; together we fall apart\nYeah, I think I'll be alright\nI'm

In [8]:
embeddings = HuggingFaceEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 57502.10it/s]


In [9]:
# download milvus script to run it in docker
# !curl -sfL https://raw.githubusercontent.com/milvus-io/milvus/master/scripts/standalone_embed.sh -o ../standalone_embed.sh
# requires sudo -
# !bash standalone_embed.sh start

In [10]:
# from pymilvus import Collection, MilvusException, MilvusClient, connections, db, utility


# connections.connect(
#     alias=client._using,
#     uri=MILVUS_URI,
#     user="root",
#     password="Milvus",
# )

# vectorstore = Milvus(
#     embedding_function=embeddings,
#     collection_name="genius-lyrics",
#     connection_args={"uri": MILVUS_URI},
# )

# try:
#     existing_databases = client.list_databases()
#     if "default" in existing_databases:
#         print(f"Database 'default' already exists.")

#         client.using_database(db_name="default")

#         collections = utility.list_collections()
#         for collection_name in collections:
#             collection = Collection(name=collection_name)
#             collection.drop()
#             print(f"Collection '{collection_name}' has been dropped.")
#     else:
#         # print(f"Database '{db_name}' does not exist.")
#         # client.create_database(db_name)
#         # client.using_database(db_name)
#         # print(f"Database '{db_name}' created successfully.")
#         pass
# except MilvusException as e:
#     print(f"An error occurred: {e}")

In [11]:
vectorstore = Chroma(
    collection_name="genius-lyrics-the-strokes",
    embedding_function=embeddings,
    persist_directory=CHROMA_PATH,

)

In [12]:
from langchain_core.documents import Document

# vectorstore.delete_collection()

In [14]:
from langchain_core.documents import Document
from tqdm import tqdm
from uuid import uuid4


def clean_metadata_value(value):
    if value is None:
        return ""
    if hasattr(value, "__class__") and value.__class__.__name__ == "NAType":
        return ""
    try:
        if value != value:  # NaN check without importing math
            return ""
    except Exception:
        pass
    return value


def build_documents_from_row(row):
    chunks = chunk_lyrics_by_section(row["lyrics"])
    # Deduplicate while preserving order
    seen = set()
    unique_chunks = []
    for chunk in chunks:
        if chunk not in seen:
            unique_chunks.append(chunk)
            seen.add(chunk)

    return [
        Document(
            page_content=chunk,
            metadata={
                "chunk_id": i,
                "song_id": clean_metadata_value(row["id"]),
                "title": clean_metadata_value(row["title"]),
                "tag": clean_metadata_value(row["tag"]),
                "artist": clean_metadata_value(row["artist"]),
                "year": clean_metadata_value(row["year"]),
                "language": clean_metadata_value(row["language"]),
            },
        )
        for i, chunk in enumerate(unique_chunks)
    ]


pbar = tqdm(total=ROWS, desc="Processing songs")
chunks_inserted = 0
songs_processed = 0
buffer: list[Document] = []

batches = lf.collect_batches(chunk_size=BATCH_SIZE)

for batch in batches:
    for row in batch.iter_rows(named=True):
        buffer.extend(build_documents_from_row(row))

        if len(buffer) >= BATCH_SIZE:
            vectorstore.add_documents(buffer)
            chunks_inserted += len(buffer)
            pbar.set_postfix(chunks=chunks_inserted)
            buffer = []

        songs_processed += 1
        pbar.update(1)

    if buffer:
        uuids = [str(uuid4()) for _ in range(len(buffer))]
        vectorstore.add_documents(buffer, ids=uuids)
        chunks_inserted += len(buffer)
        pbar.set_postfix(chunks=chunks_inserted)
        buffer = []

    # break  # remove this line to ingest the full dataset

pbar.close()

Processing songs: 100%|██████████| 101/101 [00:03<00:00, 25.32it/s, chunks=462]


In [15]:
results = vectorstore.similarity_search("aeroplane", k=20)
results

[Document(id='f2271921-6563-4a2e-9493-c934271556e6', metadata={'song_id': 5244582, 'title': 'Why Are Sundays So Depressing?', 'language': 'en', 'artist': 'The Strokes', 'tag': 'rock', 'chunk_id': 5, 'year': 2020}, page_content="[Pre-Chorus]\nI transition in\nI'm making your body wait\nLike on an aeroplane\nPlease, baby, take me away, yeah"),
 Document(id='7eef4366-4f85-4428-b27d-1e14fecae724', metadata={'song_id': 2480316, 'language': 'en', 'year': 2016, 'chunk_id': 1, 'title': 'OBLIVIUS', 'tag': 'rock', 'artist': 'The Strokes'}, page_content='[Pre-Chorus]\nTake off from the runway\nThinking of a sad day\nRacing down the highway\nLooking at me sideways'),
 Document(id='c4f5dbca-a568-4d5b-8ba4-2e569703a9ca', metadata={'title': 'OBLIVIUS Moretti Remix', 'language': 'en', 'artist': 'The Strokes', 'song_id': 2481001, 'chunk_id': 2, 'tag': 'rock', 'year': 2016}, page_content='[Pre-Chorus]\nTake off from the runway\nThinking of a sad day\nRacing down the highway\nLooking at me sideways'),
 D

In [16]:
for r in results:
    print(r.page_content, end="\n------\n")

[Pre-Chorus]
I transition in
I'm making your body wait
Like on an aeroplane
Please, baby, take me away, yeah
------
[Pre-Chorus]
Take off from the runway
Thinking of a sad day
Racing down the highway
Looking at me sideways
------
[Pre-Chorus]
Take off from the runway
Thinking of a sad day
Racing down the highway
Looking at me sideways
------
[Outro]
What side you standing on?
What side you standing on?
Ohhhh
------
[Verse 2]
I feel so different now, we trained at A-V-A
I wish you hadn't stayed, my vision's clearer now but I'm unafraid
Flying overseas, no time to feel the breeze
I took too many varieties
------
[Chorus]
A desk to organize, a product to advertise
A market to monopolize, movie stars to idolize
Leaders to scandalize, enemies to neutralize
No time to apologize, fury to tranquilize
Weapons to synchronize, cities to vapori–
------
[Verse 2]
And now the door slams shut
A child prisoner grows up
To seek his enemy's throat cut
(I'm on and on it, on and on and on it)
We're on the